In [ ]:
"""
This script is used to turn the scanpy objects into 10x cellranger output format
authors: Roy Oelen
"""

In [ ]:
# import the libraries required
import scanpy as sc
import pandas as pd
import numpy as np
import scipy.sparse as sparse
import scipy.io as sio
import scipy
import gzip
import os

In [ ]:
def scanpy_object_to_10x_format(scanpy_object_loc, output_loc):
    """
        
        Parameters
        ----------
        scanpy_object_loc : AnnData
            the AnnData Scanpy object to deconstruct
        output_loc : String
            the location of where to put the deconstructed parts
        
        Returns
        -------
        result
           0 if succesfull
    """
    # read the scanpy object
    lane_object = sc.read_h5ad(scanpy_object_loc)
    # make the genes unique
    lane_object.var_names_make_unique()
    # location of outputs
    matrix_loc = ''.join([output_loc, 'matrix.mtx.gz'])
    features_loc = ''.join([output_loc, 'features.tsv.gz'])
    barcodes_loc = ''.join([output_loc, 'barcodes.tsv.gz'])
    metadata_loc = ''.join([output_loc, 'metadata.tsv.gz'])
    # write barcodes and features
    with gzip.open(barcodes_loc, 'wb') as f:
        pd.DataFrame(data = {'barcodes' : lane_object.obs_names.tolist()}).to_csv(f, sep = '\t', header = False, index = False)
    with gzip.open(features_loc, 'wb') as f:
        pd.DataFrame(data = {'features' : lane_object.var_names.tolist()}).to_csv(f, sep = '\t', header = False, index = False)
    # write the metadata
    with gzip.open(metadata_loc, 'wb') as f:
        lane_object.obs.to_csv(f, sep = '\t', header = True, index = True, index_label = 'barcode')
    # write the matrix
    with gzip.open(matrix_loc, 'wb') as f:
        scipy.io.mmwrite(f, lane_object.X.transpose())
    # let us know we were succesfull
    return 0


def scanpy_objects_to_10x_format(scanpy_object_dir, output_dir, lanes, object_prepend='mo_', object_append='.h5ad'):
    """
        
        Parameters
        ----------
        scanpy_object_dir : String
            the location of the AnnData Scanpy objects
        output_dir : String
            the location of where to the folders of the deconstructed parts
        object_prepend : String, optional
            prepend for the h5ad object
        object_append : String, optional
            append for the h5ad object
        
        Returns
        -------
        result
           0 if succesfull
    """
    # check each lane
    for lane in lanes:
        # paste the output dir together
        lane_output_dir = ''.join([output_dir, '/', lane, '/'])
        if not os.path.exists(lane_output_dir):
           # create the directory
            os.mkdir(lane_output_dir)
        # get the location of the AnnData
        scanpy_object_loc = ''.join([scanpy_object_dir, '/', object_prepend, lane, object_append])
        if os.path.exists(scanpy_object_loc):
            # do the lane
            scanpy_object_to_10x_format(scanpy_object_loc, lane_output_dir)
        else:
            print(''.join(['skipping ', scanpy_object_loc, ', because it does not exist']))
    # let use know we were succesfull
    return 0

In [ ]:
# these are the lanes to consider
lanes = ['230105_lane1', '230105_lane2', '230105_lane3', '230105_lane4',
           '230105_lane5', '230105_lane6', '230105_lane7', '230105_lane8',
           '230112_lane1', '230112_lane2', '230112_lane3', '230112_lane4',
           '230112_lane5', '230112_lane6', '230112_lane7', '230112_lane8',
           '230120_lane1', '230120_lane2', '230120_lane3', '230120_lane4',
           '230120_lane5', '230120_lane6', '230120_lane7', '230120_lane8',
           '230127_lane1', '230127_lane2', '230127_lane3', '230127_lane4',
           '230127_lane5', '230127_lane6', '230127_lane7', '230127_lane8',
           '230202_lane1', '230202_lane2', '230202_lane3', '230202_lane4',
           '230202_lane5', '230202_lane6', '230202_lane7', '230202_lane8',
           '230209_lane1', '230209_lane2', '230209_lane3', '230209_lane4',
           '230209_lane5', '230209_lane6', '230209_lane7', '230209_lane8',
           '230216_lane1', '230216_lane2', '230216_lane3', '230216_lane4',
           '230216_lane5', '230216_lane6', '230216_lane7', '230216_lane8',
           '230223_lane1', '230223_lane2', '230223_lane3', '230223_lane4',
           '230223_lane5', '230223_lane6', '230223_lane7', '230223_lane8',
           '230302_lane1', '230302_lane2', '230302_lane3', '230302_lane4',
           '230302_lane5', '230302_lane6', '230302_lane7', '230302_lane8',
           '230316_lane1', '230316_lane2', '230316_lane3', '230316_lane4',
           '230316_lane5', '230316_lane6', '230316_lane7', '230316_lane8'
        ]

In [ ]:
# location of the scanpy objects
scanpy_objects_dir = '/groups/umcg-franke-scrna/tmp02/projects/multiome/ongoing/scanpy_preprocess_samples/objects/'
# location of the 
deconstructed_objects_dir = '/groups/umcg-franke-scrna/tmp02/projects/multiome/ongoing/scanpy_preprocess_samples/deconstructed/'

In [ ]:
# to all the lanes
scanpy_objects_to_10x_format(scanpy_objects_dir, deconstructed_objects_dir, lanes)